In [ ]:
import os
import numpy as np
import time

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator, FormatStrFormatter, FuncFormatter
import math, copy

In [ ]:
import pickle
from pathlib import Path
from typing import Any, Dict, Mapping, List

# Funcs

In [ ]:
def save_dict(d: Mapping[str, Any], file: Path, compressed: bool = False) -> None:
    """Save a mapping as a pickle (optionally gzip-compressed)."""
    file = Path(file)
    file.parent.mkdir(parents=True, exist_ok=True)

    if compressed or file.suffix == ".gz":
        import gzip
        with gzip.open(file, "wb") as f:
            pickle.dump(dict(d), f, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open(file, "wb") as f:
            pickle.dump(dict(d), f, protocol=pickle.HIGHEST_PROTOCOL)


def load_dict(file: Path) -> Dict[str, Any]:
    """Load and return a dict from a pickle (supports .gz)."""
    file = Path(file)
    if file.suffix == ".gz":
        import gzip
        with gzip.open(file, "rb") as f:
            return pickle.load(f)
    else:
        with open(file, "rb") as f:
            return pickle.load(f)

In [ ]:
def initialize_methodkey_in_dict(method_list, derivative_type_list, projection_type_list) -> List[str]:
    method_key_list = []
    if 'MeshFEM' in method_list:
        for derivative_type in derivative_type_list:
            for projection_type in projection_type_list:
                if derivative_type == 'TAD' and projection_type == 'Fbased': continue # invalid TAD and Fbased in MeshFEM
                method_key = f"MeshFEM-{derivative_type}-{projection_type}"
                method_key_list.append(method_key)

    if 'TinyAD' in method_list:
        derivative_type = 'None'
        for projection_type in projection_type_list:
            if projection_type == 'Fbased': continue  # invalid Fbased in TinyAD
            method_key = f"TinyAD-{derivative_type}-{projection_type}"
            method_key_list.append(method_key)
    
    return method_key_list

In [ ]:
def getModelNameDict():
    '''
    get a model name dict for 24 models in param benchmark
    '''
    modelNameDict = {}
    modelNameDict['cow2Disc'] = 'cow'
    modelNameDict['bumpy_sphereDisc'] = 'bumpy-sphere'
    modelNameDict['denteDisc'] = 'dente'
    modelNameDict['armadilloDisc'] = 'armadillo'
    modelNameDict['davidDisc'] = 'david'
    modelNameDict['bladeDisc'] = 'blade'
    modelNameDict['hand'] = 'hand'
    modelNameDict['gargoyle_cut'] = 'gargoyle'
    modelNameDict['vase_lion'] = 'vase-lion'
    modelNameDict['bimba100KDisc'] = 'bimba'
    modelNameDict['busteDisc'] = 'buste'
    modelNameDict['armchairDisc'] = 'armchair'
    modelNameDict['deformed_armadilloDisc'] = 'deformed-armadillo'
    modelNameDict['camille_hand100KDisc'] = 'camille-hand'
    modelNameDict['bunnyBotschDisc'] = 'bunny2'
    modelNameDict['Superman_cut2'] = 'superman2'
    modelNameDict['Superman_cut3'] = 'superman3'
    modelNameDict['Superman_cut1'] = 'superman1'
    modelNameDict['bear_cut'] = 'bear'
    modelNameDict['dragonHead2'] = 'dragon-head'
    modelNameDict['eros'] = 'eros'
    modelNameDict['buddha_cut'] = 'buddha'
    modelNameDict['Lucy_3cuts'] = 'lucy'
    modelNameDict['chinese_dragon'] = 'chinese-dragon'

    return modelNameDict

# Pre set

In [ ]:
dict_name = "total_timing_dict.pkl.gz"

In [ ]:
method_list = ["MeshFEM", "TinyAD"]

In [ ]:
Derivative_Type_List = ['AN', 'FAD', 'TAD']
Projection_Type_List = ['None', 'Fbased', 'Xbased']

# Read Dict

In [ ]:
base_path = 'DerEvalTimingExps/test_bear_macbook_1107/'
base_path = 'linux_exps/DerEvalExps/lucy_exp_MeshFEMandTinyAD_Linux/'

In [ ]:
TotalTimingDict = load_dict(os.path.join(base_path, dict_name))

In [ ]:
model_names = list(TotalTimingDict.keys())

In [ ]:
model_names

In [ ]:
print("Num of models in TotalTimingDict: ", len(model_names))

In [ ]:
query_key_str = f"{method_list[0]}-{Derivative_Type_List[0]}-{Projection_Type_List[0]}"
thread_num_list = list(TotalTimingDict[model_names[0]][query_key_str].keys())

In [ ]:
thread_num_list

# Grid Bar Plots

In [ ]:
numRows = len(model_names)
numCols = len(Projection_Type_List)
single_fig_width = 8
single_fig_height = 6
interval = 2
grid_fig_wdith = numCols * single_fig_width + (numCols - 1) * interval
grid_fig_height = numRows * single_fig_height + (numRows - 1) * interval

## Set Same Vertical Axis

In [ ]:
method_key_list = initialize_methodkey_in_dict(method_list, Derivative_Type_List, Projection_Type_List)

In [ ]:
model_dict = TotalTimingDict['Lucy_3cuts']

In [ ]:
max_timing = max([model_dict[key][1] for key in method_key_list])

In [ ]:
max_timing

In [ ]:
# manual enforce y-ticks
ymin = 0.0
ymax = 5.0

yticks = np.linspace(ymin, ymax, 11)

In [ ]:
yticks

In [ ]:
brek

## Bar Plots

In [ ]:
width = 0.2

In [ ]:
plt.figure(figsize=(grid_fig_wdith, grid_fig_height))

for model_ind, model_name in enumerate(model_names):
    model_dict = TotalTimingDict[model_name]
    
    for projection_type_ind, projection_type in enumerate(Projection_Type_List):
        ax = plt.subplot(numRows, numCols, model_ind * numCols + projection_type_ind + 1)
        if projection_type == "Fbased":  
            num_options = 2
        else:                            
            num_options = 4
        
        # Base positions for groups of bars (one per thread number)
        a = np.arange(len(thread_num_list)) * (num_options + 1) * width  # Add space between groups
        
        # get method_key_str and obtain the threading data
        method_key_ind = 0
        for method in method_list:
            if method == 'MeshFEM':  temp_derivative_type_list = Derivative_Type_List
            else:  temp_derivative_type_list = ['None']
            for derivative_type in temp_derivative_type_list:
                # skip unreasonable combinations
                if method == 'TinyAD' and projection_type == 'Fbased': continue
                if method == 'MeshFEM' and derivative_type == 'TAD' and projection_type == 'Fbased': continue
                
                method_key_str = f"{method}-{derivative_type}-{projection_type}"
                method_dict = model_dict[method_key_str]
                thread_timing_list = [method_dict[i] for i in thread_num_list]
                # Adjust positions for bars in each group
                position = a + method_key_ind * width
                label_str = f"{method}-{derivative_type}" if method=="MeshFEM" else "TinyAD"
                ax.bar(position, thread_timing_list, width=width, label=label_str)
                method_key_ind += 1
        
        # Adjust x-axis ticks to be centered
        ax.set_xticks(a + (num_options - 1) * width / 2)  # Center ticks within the group
        ax.set_xticklabels(thread_num_list)
        ax.set_xlabel("Number of Threads", fontsize=18)
        if projection_type_ind == 0:
            ax.set_ylabel("Time (s)", fontsize=18)
        ax.set_title(f"Model: {getModelNameDict()[model_name]}, Projection: {projection_type}", fontsize=20)
        
        # Ensure y-axis same
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
        ax.set_ylim(*(ymin, ymax))
        ax.set_yticks(yticks)

        ax.grid(True, linestyle='--', alpha=0.7)  # Add a grid
        ax.legend(loc='upper left')
        
plt.tight_layout()

In [ ]:
width = 0.2
MAX_OPTIONS = 4                      # the widest group across all subplots
group_stride = (MAX_OPTIONS + 1) * width

plt.figure(figsize=(grid_fig_wdith, grid_fig_height))
for model_ind, model_name in enumerate(model_names):
    model_dict = TotalTimingDict[model_name]

    for projection_type_ind, projection_type in enumerate(Projection_Type_List):
        ax = plt.subplot(numRows, numCols, model_ind * numCols + projection_type_ind + 1)

        # how many bars will be shown in this subplot
        num_options = 2 if projection_type == "Fbased" else 4

        # fixed spacing for ALL subplots
        a = np.arange(len(thread_num_list)) * group_stride     # group centers

        method_key_ind = 0
        for method in method_list:
            temp_derivative_type_list = Derivative_Type_List if method == 'MeshFEM' else ['None']
            for derivative_type in temp_derivative_type_list:
                if method == 'TinyAD' and projection_type == 'Fbased': continue
                if method == 'MeshFEM' and derivative_type == 'TAD' and projection_type == 'Fbased': continue

                # offsets that center the n bars within the fixed group slot
                offset = (method_key_ind - (num_options - 1) / 2.0) * width
                positions = a + offset

                method_key_str = f"{method}-{derivative_type}-{projection_type}"
                thread_timing_list = [model_dict[method_key_str][i] for i in thread_num_list]
                label_str = f"{method}-{derivative_type}" if method=="MeshFEM" else "TinyAD"
                ax.bar(positions, thread_timing_list, width=width, label=label_str)
                method_key_ind += 1

        # ticks at group centers; same x-limits across subplots to lock pixel width
        ax.set_xticks(a)
        ax.set_xticklabels(thread_num_list)
        ax.set_xlim(a[0] - group_stride/2, a[-1] + group_stride/2)
        ax.set_xlabel("Number of Threads", fontsize=18)
        if projection_type_ind == 0:
            ax.set_ylabel("Time (s)", fontsize=18)
        ax.set_title(f"Model: {getModelNameDict()[model_name]}, Projection: {projection_type}", fontsize=20)

        # Ensure y-axis same
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
        ax.set_ylim(*(ymin, ymax))
        ax.set_yticks(yticks)

        ax.grid(True, linestyle='--', alpha=0.7)  # Add a grid
        ax.legend(loc='upper right')
        
plt.tight_layout()
# plt.savefig(f'PaperFigures/{getModelNameDict()[model_name]}_derivative_timing_barplots.png', dpi=500, bbox_inches="tight", pad_inches=0)